In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *

print(LOB_LIST)

['Technology Resiliency', 'Payments', 'Customer Platforms', 'Risk & Compliance', 'Treasury & Finance', 'Internal Services']


In [3]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import date

fake = Faker()

NUM_APPS = 1000

In [4]:
first_names = [fake.first_name() for _ in range(200)]
last_names = [fake.last_name() for _ in range(200)]

def make_name():
    return f"{random.choice(first_names)} {random.choice(last_names)}"

resiliency_leads = list({make_name() for _ in range(24)})
app_owners = list({make_name() for _ in range(120)})

print("Resiliency leads:", len(resiliency_leads))
print("App owners:", len(app_owners))
print(resiliency_leads[:5])
print(app_owners[:5])

Resiliency leads: 24
App owners: 119
['Anna Johnson', 'Jennifer Hill', 'Jessica Simpson', 'Nicholas Yoder', 'Michelle Webb']
['Matthew Miller', 'Jamie Thomas', 'Heather Johnson', 'Pedro Combs', 'Jacqueline Boyd']


In [5]:
def weighted_choice(options, weights):
    return random.choices(options, weights=weights, k=1)[0]

In [6]:
LOB_WEIGHTS = [16, 18, 20, 17, 14, 15]

APP_STATUS_WEIGHTS = [84, 8, 5, 2, 1]

DEPLOY_STATUS_WEIGHTS = [68, 14, 14, 4]

HOSTING_WEIGHTS = [58, 24, 10, 8]

OS_FAMILY_WEIGHTS = [48, 33, 9, 10]

LAST_TEST_RESULT_OPTIONS = ["Pass", "Pass with Issues", "Fail", "Not Recent"]
LAST_TEST_RESULT_WEIGHTS = [74, 16, 6, 4]

IN_SCOPE_OPTIONS = ["Y", "N"]
IN_SCOPE_WEIGHTS = [78, 22]

ENV_OPTIONS = ["Production", "Mixed", "QA", "Dev"]
ENV_WEIGHTS = [60, 20, 10, 10]

In [7]:
RTO_SCORES = list(RTO_MAP.keys())
RTO_WEIGHTS = [4, 6, 9, 11, 15, 16, 15, 11, 8, 5]

In [8]:
def make_email(full_name, domain=EMAIL_DOMAIN):
    return full_name.lower().replace(" ", ".") + f"@{domain}"

owner_emails = {owner: make_email(owner) for owner in app_owners}
list(owner_emails.items())[:5]

[('Matthew Miller', 'matthew.miller@northstarfinancial.com'),
 ('Jamie Thomas', 'jamie.thomas@northstarfinancial.com'),
 ('Heather Johnson', 'heather.johnson@northstarfinancial.com'),
 ('Pedro Combs', 'pedro.combs@northstarfinancial.com'),
 ('Jacqueline Boyd', 'jacqueline.boyd@northstarfinancial.com')]

In [9]:
app_name_prefixes = [
    "Atlas", "Nimbus", "Summit", "Orion", "Harbor", "Pinnacle", "Vertex",
    "Meridian", "Sterling", "Cobalt", "Granite", "Catalyst", "Sentinel",
    "Evergreen", "Falcon", "Beacon", "NorthStar", "Apex", "Horizon", "Keystone"
]

app_name_suffixes = [
    "Payments", "Gateway", "Portal", "Engine", "Hub", "Manager", "Tracker",
    "Console", "Services", "Workbench", "Coordinator", "Monitor", "Registry",
    "Platform", "Exchange", "Insights", "Scheduler", "Operations", "Recovery", "Control"
]

def make_app_name():
    return f"{random.choice(app_name_prefixes)} {random.choice(app_name_suffixes)}"

[make_app_name() for _ in range(10)]

['Vertex Payments',
 'Granite Monitor',
 'Catalyst Hub',
 'Keystone Tracker',
 'Meridian Console',
 'Meridian Scheduler',
 'Orion Portal',
 'Evergreen Platform',
 'Falcon Payments',
 'Pinnacle Operations']

In [10]:
apps_data = []

for i in range(1, NUM_APPS + 1):
    app_id = f"APP-{i:04d}"
    app_name = make_app_name()
    app_owner = random.choice(app_owners)
    owner_team = random.choice(OWNER_TEAMS)
    resiliency_lead = random.choice(resiliency_leads)
    lob = weighted_choice(LOB_LIST, LOB_WEIGHTS)
    app_status = weighted_choice(APP_STATUS, APP_STATUS_WEIGHTS)
    deploy_status = weighted_choice(DEPLOY_STATUS, DEPLOY_STATUS_WEIGHTS)
    in_scope = weighted_choice(IN_SCOPE_OPTIONS, IN_SCOPE_WEIGHTS)
    rto_score = weighted_choice(RTO_SCORES, RTO_WEIGHTS)
    rto_hours = RTO_MAP[rto_score]
    primary_dc = random.choice(DATACENTERS)

    secondary_dc_candidates = [dc for dc in DATACENTERS if dc != primary_dc]
    secondary_dc = random.choice(secondary_dc_candidates) if random.random() < 0.72 else None

    env = weighted_choice(ENV_OPTIONS, ENV_WEIGHTS)
    hosting = weighted_choice(HOSTING_TYPES, HOSTING_WEIGHTS)
    os_family = weighted_choice(OS_FAMILIES, OS_FAMILY_WEIGHTS)

    last_test_date = fake.date_between(start_date="-18M", end_date="today")
    last_test_result = weighted_choice(LAST_TEST_RESULT_OPTIONS, LAST_TEST_RESULT_WEIGHTS)

    apps_data.append({
        "app_id": app_id,
        "app_name": app_name,
        "app_owner": app_owner,
        "owner_email": owner_emails[app_owner],
        "owner_team": owner_team,
        "resiliency_lead": resiliency_lead,
        "lob": lob,
        "app_status": app_status,
        "deploy_status": deploy_status,
        "in_scope": in_scope,
        "rto_score": rto_score,
        "rto_hours": rto_hours,
        "primary_dc": primary_dc,
        "secondary_dc": secondary_dc,
        "env": env,
        "hosting": hosting,
        "os_family": os_family,
        "last_test_date": last_test_date,
        "last_test_result": last_test_result
    })

apps_df = pd.DataFrame(apps_data)
apps_df.head()

,app_id,app_name,app_owner,owner_email,owner_team,resiliency_lead,lob,app_status,deploy_status,in_scope,rto_score,rto_hours,primary_dc,secondary_dc,env,hosting,os_family,last_test_date,last_test_result
0,APP-0001,Granite Operations,Ryan Weaver,ryan.weaver@northstarfinancial.com,Digital Platforms,Amy Brown,Customer Platforms,Active,Production,Y,5,8.0,Dallas,Des Moines,Production,On-Prem,Mixed,2024-12-21,Pass with Issues
1,APP-0002,Harbor Recovery,Stacy Huang,stacy.huang@northstarfinancial.com,Digital Platforms,Anthony Davis,Risk & Compliance,Retiring,Decommissioning,Y,8,48.0,Des Moines,St. Louis,Production,On-Prem,Mixed,2024-11-04,Pass
2,APP-0003,Atlas Console,Christopher Dickerson,christopher.dickerson@northstarfinancial.com,Treasury Technology,Mark Wong,Technology Resiliency,Decommission Planned,Hybrid,Y,8,48.0,Columbus,Minneapolis,Production,On-Prem,Windows,2025-04-05,Pass
3,APP-0004,Beacon Services,Elizabeth Brooks,elizabeth.brooks@northstarfinancial.com,Middleware Services,Jessica Simpson,Customer Platforms,Active,Non-Production,N,2,1.0,Ashburn,Charlotte,Production,Hybrid Cloud,Linux,2026-02-13,Pass
4,APP-0005,Granite Insights,Kenneth Moon,kenneth.moon@northstarfinancial.com,Treasury Technology,Christy Miller,Risk & Compliance,Active,Hybrid,Y,7,24.0,Charlotte,Phoenix,Production,Hybrid Cloud,Unix,2025-09-03,Pass


In [11]:
print(apps_df.shape)
print("\nIn-scope counts:")
print(apps_df["in_scope"].value_counts(dropna=False))

print("\nLOB counts:")
print(apps_df["lob"].value_counts())

print("\nDeploy status counts:")
print(apps_df["deploy_status"].value_counts())

print("\nRTO score counts:")
print(apps_df["rto_score"].value_counts().sort_index())

(1000, 19)

In-scope counts:
in_scope
Y    750
N    250
Name: count, dtype: int64

LOB counts:
lob
Customer Platforms       205
Payments                 195
Risk & Compliance        165
Technology Resiliency    150
Treasury & Finance       145
Internal Services        140
Name: count, dtype: int64

Deploy status counts:
deploy_status
Production         709
Hybrid             137
Non-Production     134
Decommissioning     20
Name: count, dtype: int64

RTO score counts:
rto_score
1      55
2      75
3      83
4     118
5     129
6     171
7     141
8     114
9      77
10     37
Name: count, dtype: int64


In [12]:
apps_df.isnull().sum()

app_id                0
app_name              0
app_owner             0
owner_email           0
owner_team            0
resiliency_lead       0
lob                   0
app_status            0
deploy_status         0
in_scope              0
rto_score             0
rto_hours             0
primary_dc            0
secondary_dc        302
env                   0
hosting               0
os_family             0
last_test_date        0
last_test_result      0
dtype: int64

In [13]:
PRODUCT_CATALOG = {
    "Operating System": {
        "RHEL": ["7.9", "8.4", "8.6", "8.8", "9.1"],
        "Windows Server": ["2016 build 14393", "2019 build 17763", "2022 build 20348"],
        "Ubuntu Server": ["20.04.6", "22.04.3"],
        "SUSE Linux": ["12 SP5", "15 SP4"]
    },
    "Database": {
        "Oracle Database": ["19.13.0", "19.18.0", "19.20.0"],
        "SQL Server": ["2017 CU31", "2019 CU22", "2022 CU8"],
        "PostgreSQL": ["13.11", "14.6", "15.4"],
        "MongoDB Enterprise": ["5.0.18", "6.0.9"]
    },
    "Middleware": {
        "WebLogic": ["12.2.1.3", "12.2.1.4"],
        "WebSphere": ["9.0.5.7", "9.0.5.14"],
        "Tomcat": ["9.0.73", "9.0.82", "10.1.18"],
        "JBoss EAP": ["7.3.0", "7.4.10"]
    },
    "Web Server / API": {
        "Apache HTTP Server": ["2.4.52", "2.4.57"],
        "NGINX": ["1.20.2", "1.24.0"],
        "Apigee Gateway": ["4.52.00", "4.53.01"],
        "IIS": ["10.0.17763", "10.0.20348"]
    },
    "Security / Identity": {
        "Okta Agent": ["3.6.0", "3.8.1"],
        "CyberArk": ["12.2.4", "13.0.2"],
        "PingFederate": ["11.2.0", "12.0.1"],
        "SailPoint IQService": ["8.2", "8.4"]
    },
    "Messaging / Integration": {
        "IBM MQ": ["9.2.0", "9.3.2"],
        "Kafka": ["2.8.2", "3.5.1"],
        "MuleSoft Runtime": ["4.4.0", "4.6.3"],
        "TIBCO EMS": ["8.6.0", "8.7.1"]
    },
    "Monitoring / Agent": {
        "Splunk UF": ["9.0.4", "9.1.1"],
        "Dynatrace OneAgent": ["1.265", "1.281"],
        "AppDynamics Agent": ["22.11.0", "23.6.0"],
        "Tanium Client": ["7.4.9", "7.6.1"]
    }
}

In [14]:
def pick_product():
    category = random.choice(list(PRODUCT_CATALOG.keys()))
    product = random.choice(list(PRODUCT_CATALOG[category].keys()))
    versions = PRODUCT_CATALOG[category][product]
    return category, product, versions

In [15]:
def generate_versions(versions):
    approved = versions[-1]  # latest version
    
    # 70% slightly behind, 30% very outdated
    if random.random() < 0.7:
        detected = random.choice(versions[:-1])
    else:
        detected = versions[0]
    
    return approved, detected

In [16]:
apps_with_drift = apps_df.sample(n=280, random_state=42)

In [17]:
year_counters = {}
drift_records = []

# LOB short codes for instance naming
LOB_CODES = {
    "Technology Resiliency": "RES",
    "Payments": "PAY",
    "Customer Platforms": "CUS",
    "Risk & Compliance": "RSK",
    "Treasury & Finance": "TRS",
    "Internal Services": "INT"
}

# Product short codes for instance naming
PRODUCT_CODES = {
    "RHEL": "RHEL", "Windows Server": "WIN", "Ubuntu Server": "UBU", "SUSE Linux": "SUS",
    "Oracle Database": "ORA", "SQL Server": "SQL", "PostgreSQL": "PG", "MongoDB Enterprise": "MDB",
    "WebLogic": "WL", "WebSphere": "WS", "Tomcat": "TC", "JBoss EAP": "JB",
    "Apache HTTP Server": "APA", "NGINX": "NGX", "Apigee Gateway": "APG", "IIS": "IIS",
    "Okta Agent": "OKT", "CyberArk": "CYA", "PingFederate": "PNG", "SailPoint IQService": "SPT",
    "IBM MQ": "MQ", "Kafka": "KFK", "MuleSoft Runtime": "MUL", "TIBCO EMS": "TIB",
    "Splunk UF": "SPL", "Dynatrace OneAgent": "DYN", "AppDynamics Agent": "APD", "Tanium Client": "TAN"
}

ENV_CODES = {
    "Production": "PRD", "Pre-Production": "PPD", "QA": "QAT", "Dev": "DEV"
}

instance_counters = {}

for _, app in apps_with_drift.iterrows():

    num_drifts = np.random.choice([1, 2, 3, 4], p=[0.58, 0.24, 0.11, 0.07])

    for _ in range(num_drifts):

        category, product, versions = pick_product()
        approved, detected = generate_versions(versions)

        open_date = fake.date_between(start_date="-12M", end_date="-10d")
        open_year = open_date.year

        year_counters.setdefault(open_year, 0)
        year_counters[open_year] += 1
        drift_id = f"DR-{open_year}-{year_counters[open_year]:06d}"

        environment = weighted_choice(ENVIRONMENTS, [63, 17, 12, 8])

        # Build instance name
        lob_code = LOB_CODES.get(app["lob"], "GEN")
        prod_code = PRODUCT_CODES.get(product, "APP")
        env_code = ENV_CODES.get(environment, "PRD")
        inst_key = f"{lob_code}{prod_code}{env_code}"
        instance_counters.setdefault(inst_key, 0)
        instance_counters[inst_key] += 1
        instance_name = f"{inst_key}{instance_counters[inst_key]:02d}"

        if random.random() < 0.62:
            close_date = fake.date_between(start_date=open_date, end_date="today")
            status = "Closed"
            duration = (close_date - open_date).days
        else:
            close_date = None
            status = "Open"
            duration = (date.today() - open_date).days

        exemption_requested = "Y" if random.random() < 0.28 else "N"

        if exemption_requested == "Y":
            exemption_result = np.random.choice(
                ["Approved", "Pending", "Denied"],
                p=[0.44, 0.34, 0.22]
            )
        else:
            exemption_result = None

        drift_records.append({
            "drift_id": drift_id,
            "app_id": app["app_id"],
            "lob": app["lob"],
            "rto_score": app["rto_score"],
            "rto_hours": app["rto_hours"],
            "resiliency_lead": app["resiliency_lead"],
            "primary_dc": app["primary_dc"],
            "secondary_dc": app["secondary_dc"],
            "product_category": category,
            "product": product,
            "instance_name": instance_name,
            "approved_version": approved,
            "detected_version": detected,
            "status": status,
            "drift_open_date": open_date,
            "drift_close_date": close_date,
            "drift_duration_days": duration,
            "exemption_requested": exemption_requested,
            "exemption_result": exemption_result,
            "root_cause": random.choice(ROOT_CAUSES),
            "environment": environment
        })

drift_df = pd.DataFrame(drift_records)
drift_df.head()

,drift_id,app_id,lob,rto_score,rto_hours,resiliency_lead,primary_dc,secondary_dc,product_category,product,...,approved_version,detected_version,status,drift_open_date,drift_close_date,drift_duration_days,exemption_requested,exemption_result,root_cause,environment
0,DR-2025-000001,APP-0522,Payments,5,8.0,Nicole Johnson,Phoenix,Charlotte,Security / Identity,PingFederate,...,12.0.1,11.2.0,Open,2025-05-11,None,327,N,NaN,Patch not tested,Pre-Production
1,DR-2025-000002,APP-0522,Payments,5,8.0,Nicole Johnson,Phoenix,Charlotte,Web Server / API,IIS,...,10.0.20348,10.0.17763,Open,2025-07-19,None,258,N,NaN,False positive detection,Production
2,DR-2025-000003,APP-0522,Payments,5,8.0,Nicole Johnson,Phoenix,Charlotte,Monitoring / Agent,Dynatrace OneAgent,...,1.281,1.265,Closed,2025-05-26,2026-03-07,285,N,NaN,Infrastructure constraint,Production
3,DR-2026-000001,APP-0738,Customer Platforms,5,8.0,Cheryl Brown,Columbus,Phoenix,Security / Identity,PingFederate,...,12.0.1,11.2.0,Open,2026-03-19,None,15,N,NaN,False positive detection,Production
4,DR-2026-000002,APP-0741,Treasury & Finance,9,72.0,Caitlin Baldwin,Minneapolis,Dallas,Operating System,Ubuntu Server,...,22.04.3,20.04.6,Closed,2026-03-22,2026-03-26,4,N,NaN,Vendor compatibility issue,Production


In [18]:
print(drift_df.shape)
print("\nColumns:", drift_df.columns.tolist())
print("\nSample instance names:")
print(drift_df["instance_name"].sample(10).tolist())
print("\nDatacenter sample:")
print(drift_df[["drift_id", "primary_dc", "secondary_dc", "lob", "rto_score"]].head(8))

(475, 21)

Columns: ['drift_id', 'app_id', 'lob', 'rto_score', 'rto_hours', 'resiliency_lead', 'primary_dc', 'secondary_dc', 'product_category', 'product', 'instance_name', 'approved_version', 'detected_version', 'status', 'drift_open_date', 'drift_close_date', 'drift_duration_days', 'exemption_requested', 'exemption_result', 'root_cause', 'environment']

Sample instance names:
['PAYWSPRD02', 'RESIISPPD01', 'RSKIISPRD01', 'PAYSQLPRD02', 'RSKJBPRD01', 'PAYWLPRD02', 'PAYKFKPRD04', 'TRSSPLPRD01', 'CUSMQQAT01', 'INTSQLPRD01']

Datacenter sample:
         drift_id   primary_dc secondary_dc                 lob  rto_score
0  DR-2025-000001      Phoenix    Charlotte            Payments          5
1  DR-2025-000002      Phoenix    Charlotte            Payments          5
2  DR-2025-000003      Phoenix    Charlotte            Payments          5
3  DR-2026-000001     Columbus      Phoenix  Customer Platforms          5
4  DR-2026-000002  Minneapolis       Dallas  Treasury & Finance          9
5 

In [19]:
print(drift_df.shape)

print("\nStatus counts:")
print(drift_df["status"].value_counts())

print("\nExemption requested counts:")
print(drift_df["exemption_requested"].value_counts())

print("\nExemption result counts:")
print(drift_df["exemption_result"].value_counts(dropna=False))

print("\nProduct category counts:")
print(drift_df["product_category"].value_counts())

print("\nAny approved == detected?")
print((drift_df["approved_version"] == drift_df["detected_version"]).sum())


(475, 21)

Status counts:
status
Closed    312
Open      163
Name: count, dtype: int64

Exemption requested counts:
exemption_requested
N    346
Y    129
Name: count, dtype: int64

Exemption result counts:
exemption_result
NaN         346
Approved     58
Pending      44
Denied       27
Name: count, dtype: int64

Product category counts:
product_category
Security / Identity        72
Operating System           72
Monitoring / Agent         70
Middleware                 69
Messaging / Integration    66
Web Server / API           64
Database                   62
Name: count, dtype: int64

Any approved == detected?
0


In [20]:
drift_df["drift_id"].head(20)

0     DR-2025-000001
1     DR-2025-000002
2     DR-2025-000003
3     DR-2026-000001
4     DR-2026-000002
5     DR-2025-000004
6     DR-2025-000005
7     DR-2025-000006
8     DR-2025-000007
9     DR-2025-000008
10    DR-2025-000009
11    DR-2026-000003
12    DR-2025-000010
13    DR-2026-000004
14    DR-2025-000011
15    DR-2025-000012
16    DR-2025-000013
17    DR-2025-000014
18    DR-2025-000015
19    DR-2025-000016
Name: drift_id, dtype: str

In [21]:
import textwrap

NOTE_TEMPLATES = {
    "Exemption Request": [
        "Upgrade to approved version deferred pending validation of downstream payment gateway compatibility.",
        "Application team requested temporary exception due to regression risk during quarter-end processing window.",
        "Approved version introduces authentication failures in lower environment; requesting extension until vendor review is complete.",
        "Upgrade postponed due to dependency on shared middleware patch scheduled for next maintenance cycle.",
        "Application is targeted for retirement in Q4; team requested exemption through decommission milestone.",
        "New version conflicts with internal reporting package used by Treasury operations; exemption requested pending fix.",
        "Vendor patch validated in lower environment but production deployment blocked pending downstream API certification.",
    ],
    "Remediation Update": [
        "App owner confirmed patch testing will begin during next approved maintenance window.",
        "Infrastructure team completed lower-environment validation; production deployment remains pending CAB approval.",
        "Dependency on Oracle client upgrade remains unresolved; no production remediation date confirmed.",
        "Engineering team reported successful QA validation and is preparing production rollout plan.",
        "Remediation target date revised; infrastructure team identified additional dependency on shared load balancer config.",
        "Owner confirmed coordination with platform engineering team to schedule upgrade during low-traffic window.",
    ],
    "Escalation": [
        "Drift exceeds 90 days for critical production application. Escalation sent to resiliency lead.",
        "Pending exemption has remained unresolved beyond review SLA. Follow-up requested from compliance.",
        "Multiple open drift instances identified for same application owner; bulk remediation discussion recommended.",
        "No owner response received after two outreach attempts. Escalating to owner team manager.",
        "Critical application approaching 120-day threshold with no remediation plan on file.",
    ],
    "Owner Update": [
        "Application owner confirmed awareness of drift and is coordinating with vendor on patch schedule.",
        "Owner indicated upgrade will be bundled with Q2 platform refresh currently in planning.",
        "Team lead confirmed remediation is blocked pending security review of new version.",
        "Owner requested additional time; dependency on third-party certification process not yet complete.",
    ],
    "Validation Result": [
        "Patch applied successfully in QA environment. Production deployment scheduled for next maintenance window.",
        "Version upgrade validated; no adverse impact detected in pre-production testing.",
        "Validation failed in lower environment due to SSL handshake error with upstream connector. Rollback initiated.",
        "Compatibility testing complete. Approved version confirmed stable with current application stack.",
    ],
    "Closure Note": [
        "Detected version aligned to approved standard during scheduled maintenance activity.",
        "Drift closed after remediation validation confirmed in production.",
        "Record closed after false positive determination from updated discovery scan.",
        "Upgrade completed during quarterly maintenance window. No issues reported post-deployment.",
    ],
    "Exemption Review": [
        "Compliance team reviewed exemption request. Additional documentation required before approval.",
        "Exemption approved pending confirmation of remediation target date from application owner.",
        "Exemption denied; risk accepted without compensating controls is outside policy threshold.",
        "Review complete. Exemption extended 60 days pending vendor patch availability.",
    ]
}

note_year_counters = {}
note_records = []

for _, drift in drift_df.iterrows():
    
    # Decide how many notes this drift gets
    num_notes = np.random.choice([0, 1, 2, 3, 4], p=[0.20, 0.28, 0.24, 0.16, 0.12])
    
    if num_notes == 0:
        continue
    
    for n in range(num_notes):
        
        # Weight note types based on drift status and exemption
        if drift["exemption_requested"] == "Y":
            note_type = np.random.choice(
                list(NOTE_TEMPLATES.keys()),
                p=[0.25, 0.20, 0.15, 0.15, 0.10, 0.08, 0.07]
            )
        elif drift["status"] == "Closed":
            note_type = np.random.choice(
                list(NOTE_TEMPLATES.keys()),
                p=[0.05, 0.25, 0.10, 0.15, 0.20, 0.20, 0.05]
            )
        else:
            note_type = np.random.choice(
                list(NOTE_TEMPLATES.keys()),
                p=[0.10, 0.30, 0.20, 0.20, 0.10, 0.05, 0.05]
            )
        
        note_text = random.choice(NOTE_TEMPLATES[note_type])
        note_date = fake.date_between(
            start_date=drift["drift_open_date"],
            end_date=drift["drift_close_date"] if drift["drift_close_date"] else date.today()
        )
        note_year = note_date.year
        note_year_counters.setdefault(note_year, 0)
        note_year_counters[note_year] += 1
        note_id = f"NOTE-{note_year}-{note_year_counters[note_year]:06d}"
        
        note_records.append({
            "note_id": note_id,
            "drift_id": drift["drift_id"],
            "app_id": drift["app_id"],
            "note_date": note_date,
            "note_type": note_type,
            "author_role": random.choice(AUTHOR_ROLES),
            "note_text": note_text,
            "escalation_flag": 1 if note_type == "Escalation" else 0
        })
        

notes_df = pd.DataFrame(note_records)
notes_df.head(10)

,note_id,drift_id,app_id,note_date,note_type,author_role,note_text,escalation_flag
0,NOTE-2025-000001,DR-2025-000002,APP-0522,2025-08-12,Owner Update,Infrastructure Engineer,Application owner confirmed awareness of drift...,0
1,NOTE-2025-000002,DR-2025-000003,APP-0522,2025-08-06,Validation Result,Platform Engineer,Patch applied successfully in QA environment. ...,0
2,NOTE-2025-000003,DR-2025-000003,APP-0522,2025-10-17,Validation Result,Compliance Analyst,Validation failed in lower environment due to ...,0
3,NOTE-2025-000004,DR-2025-000004,APP-0741,2025-12-09,Exemption Request,Resiliency Lead,Application team requested temporary exception...,0
4,NOTE-2026-000001,DR-2025-000004,APP-0741,2026-01-28,Escalation,Resiliency Lead,Pending exemption has remained unresolved beyo...,1
5,NOTE-2026-000002,DR-2025-000005,APP-0741,2026-01-13,Validation Result,Resiliency Lead,Version upgrade validated; no adverse impact d...,0
6,NOTE-2026-000003,DR-2025-000005,APP-0741,2026-03-08,Owner Update,Platform Engineer,Team lead confirmed remediation is blocked pen...,0
7,NOTE-2025-000005,DR-2025-000006,APP-0661,2025-10-09,Closure Note,Platform Engineer,Drift closed after remediation validation conf...,0
8,NOTE-2025-000006,DR-2025-000006,APP-0661,2025-10-09,Exemption Request,Infrastructure Engineer,Upgrade postponed due to dependency on shared ...,0
9,NOTE-2025-000007,DR-2025-000006,APP-0661,2025-10-21,Validation Result,Application Owner,Patch applied successfully in QA environment. ...,0


In [22]:
print(notes_df.shape)

print("\nNote types:")
print(notes_df["note_type"].value_counts())

print("\nNote ID year distribution:")
print(notes_df["note_id"].str[5:9].value_counts().sort_index())

print("\nEscalation flags:")
print(notes_df["escalation_flag"].value_counts())

print("\nUnique IDs check:")
print(notes_df["note_id"].is_unique)

print("\nSample notes:")
print(notes_df[["note_id", "drift_id", "note_type", "note_text"]].sample(5).to_string())

(804, 8)

Note types:
note_type
Remediation Update    194
Owner Update          152
Validation Result     122
Escalation            110
Closure Note          100
Exemption Request      91
Exemption Review       35
Name: count, dtype: int64

Note ID year distribution:
note_id
2025    436
2026    368
Name: count, dtype: int64

Escalation flags:
escalation_flag
0    694
1    110
Name: count, dtype: int64

Unique IDs check:
True

Sample notes:
              note_id        drift_id          note_type                                                                                                   note_text
90   NOTE-2025-000053  DR-2025-000043       Owner Update                          Team lead confirmed remediation is blocked pending security review of new version.
341  NOTE-2026-000157  DR-2025-000148       Closure Note                               Record closed after false positive determination from updated discovery scan.
436  NOTE-2025-000233  DR-2025-000188  Validation Result     

In [23]:
import sqlite3

db_path = "../data/processed/northstar_drift.db"

conn = sqlite3.connect(db_path)

apps_df.to_sql("applications", conn, if_exists="replace", index=False)
drift_df.to_sql("drift_instances", conn, if_exists="replace", index=False)
notes_df.to_sql("drift_notes", conn, if_exists="replace", index=False)

conn.close()

print("Database written successfully.")
print(f"Location: {db_path}")

Database written successfully.
Location: ../data/processed/northstar_drift.db


In [25]:
conn = sqlite3.connect(db_path)

for table in ["applications", "drift_instances", "drift_notes"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

conn.close()

applications: 1000 rows
drift_instances: 475 rows
drift_notes: 804 rows


In [26]:
apps_df.to_csv("../data/processed/applications.csv", index=False)
drift_df.to_csv("../data/processed/drift_instances.csv", index=False)
notes_df.to_csv("../data/processed/drift_notes.csv", index=False)

print("CSVs exported.")

CSVs exported.
